In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def U():
    """
    U function that creates the circuit that checks the parity of the number of forests.
    The inputs are already embedded via BasisEmbedding on wires 0-7. The parity output is wire 8.
    """
    # If wire 0 is 1, that starts a forest, so flip wire 8
    qml.CNOT(wires=[0, 8])

    # For each i=1..7: a forest starts at i if wire[i]==1 and wire[i-1]==0
    for i in range(1, 8):
        # We want: parity flips if wire[i]==1 and wire[i-1]==0
        # Implement using Toffoli with a "0" control on wire[i-1]: surround with X
        qml.PauliX(wires=i-1)
        qml.Toffoli(wires=[i, i-1, 8])
        qml.PauliX(wires=i-1)
    # Result: parity of forest count encoded on wire 8


def run(test_case_input: str) -> str:
    input_data = json.loads(test_case_input)
    wires_input = [0,1,2,3,4,5,6,7]

    dev = qml.device("default.qubit", wires=9)

    @qml.qnode(dev)
    def circuit():
        qml.BasisEmbedding(input_data, wires=wires_input)
        U()
        return qml.probs(wires=8)

    # Since it's deterministic, probability is either 1 at index 0 or 1 at index 1.
    probs = circuit()
    result = int(np.round(probs[1]))
    return str(result)

def check(have: str, want: str) -> None:
    assert np.isclose(float(have), float(want)), f"Wrong answer! Have: {have}, Want: {want}"

# These are the public test cases
test_cases = [
    ('[1,0,1,1,0,1,1,1]', '1'),
    ('[0,0,0,0,0,1,0,1]', '0'),
]

# This will run the public test cases locally
def test():
    for i, (input_, expected_output) in enumerate(test_cases):
        print(f"Running test case {i} with input '{input_}'...")

        try:
            output = run(input_)
        except Exception as exc:
            print(f"Runtime Error. {exc}")
        else:
            try:
                check(output, expected_output)
            except Exception as exc:
                print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
            else:
                print("Correct!")

if __name__ == "__main__":
    test()


Running test case 0 with input '[1,0,1,1,0,1,1,1]'...
Correct!
Running test case 1 with input '[0,0,0,0,0,1,0,1]'...
Correct!
